In [ ]:
import osmnx as ox
import networkx as nx
import igraph as ig
import pandas as pd
import geopandas as gpd
import time
import matplotlib.pyplot as plt

***

Import Copenhagen street network as example data

In [ ]:
# download some random network
G = ox.graph_from_place(
    query="Copenhagen Municipality",
    network_type='all',
    simplify=True, 
    retain_all=False,
    truncate_by_edge=False,
    which_result=None,
    custom_filter=None
)

In [ ]:
print("Edges:", len(G.edges))
print("Nodes:", len(G.nodes))

In [ ]:
nodes, edges = ox.graph_to_gdfs(G, nodes = True, edges = True)

***

select source and target node for example path computation

In [ ]:
fig, ax = plt.subplots(1,1)
edges.plot(ax=ax, linewidth=1, zorder=0, alpha = 0.1)
nodes.iloc[[25,1225]].plot(ax=ax, color = "red", zorder = 1)

In [ ]:
nodes.iloc[[25,1225]]

In [ ]:
# OSMIDS of these nodes are "source" and "target" IDs in networkx
s_nx = 118820
t_nx = 8091688

***

convert to Graph

In [ ]:
# convert from MultiDiGraph to Graph
G = nx.Graph(G)

In [ ]:
print("Edges:", len(G.edges))
print("Nodes:", len(G.nodes))

***

Time shortest path computations for selected source & target with **networkx**

In [ ]:
%%timeit
nx.shortest_path(
    G=G,
    source=s_nx,
    target=t_nx
)
# shortest path (unweighted)

In [ ]:
%%timeit
nx.shortest_path(
    G=G,
    source=s_nx,
    target=t_nx,
    weight="length"
)
# shortest path (weighted)

In [ ]:
%%timeit
i = 0
for path in nx.shortest_simple_paths(
    G=G,
    source=s_nx,
    target=t_nx,
    #weight="length"
):
    #print(len(path))
    i += 1
    if i > 1:
        break
# k shortest paths unweighted, k = 2

In [ ]:
%%timeit
i = 0
for path in nx.shortest_simple_paths(
    G=G,
    source=s_nx,
    target=t_nx,
    weight="length"
):
    #print(len(path))
    i += 1
    if i > 1:
        break
# k shortest paths weighted, k = 2

***

Time shortest path computations for selected source & target with **igraph**

In [ ]:
# convert from networkx to igraph
g = ig.Graph.from_networkx(G, vertex_attr_hashable="osmid")
# get corresponding node ids in igraph
s_ig, t_ig = [v.index for v in g.vs.select(osmid_in=[s_nx,t_nx])]

In [ ]:
%%timeit
g.get_shortest_path(
    v=s_ig,
    to=t_ig,
    weights=None,
    mode='out',
    output='vpath',
    algorithm='auto'
)
# shortest path (unweighted)
# ~15x faster than nx

In [ ]:
%%timeit
g.get_shortest_path(
    v=s_ig,
    to=t_ig,
    weights="length",
    mode='out',
    output='vpath',
    algorithm='auto'
)
# shortest path (weighted)
## 6x faster than nx

In [ ]:
%%timeit
l = g.get_k_shortest_paths(
    v=s_ig,
    to=t_ig,
    k = 2,
    #weights="length",
    #mode='out',
    #output="vpath"
)
# k shortest paths (unweighted), k = 2

In [ ]:
%%timeit
l = g.get_k_shortest_paths(
    v=s_ig,
    to=t_ig,
    k = 2,
    weights="length",
    #mode='out',
    #output="vpath"
)
# k shortest paths (weighted), k = 2

***

How to get a list of nx node ids from list of igraph node ids:

In [ ]:
ig_nodelist = list(range(100)) # random nodelist
nx_nodelist = g.vs[ig_nodelist]["osmid"] # query by ig index, then nx attribute name